In [ ]:
# Step 1: Upload dataset (tags.csv)

from google.colab import files
uploaded = files.upload()

Saving tags_cleaned.csv to tags_cleaned (1).csv


In [ ]:
# Step 2: Import libraries

import pandas as pd
import numpy as np

# NLP library (for initial labeling)
from textblob import TextBlob

# Machine Learning libraries
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# Step 3: Load cleaned dataset

tags = pd.read_csv("tags_cleaned.csv")

# Display sample data
print("Sample Data:")
print(tags.head())

# Check structure
print("\nDataset Info:")
print(tags.info())

Sample Data:
   userId  movieId                tag              date  Unnamed: 4  \
0       2    89774          tom hardy  24-10-2015 19:33         NaN   
1       2    89774       boxing story  24-10-2015 19:33         NaN   
2       2   106782    martin scorsese  24-10-2015 19:30         NaN   
3       2   106782  leonardo dicaprio  24-10-2015 19:30         NaN   
4       2   106782              drugs  24-10-2015 19:30         NaN   

   Unnamed: 5  Unnamed: 6  Unnamed: 7  Unnamed: 8  Unnamed: 9  
0         NaN         NaN         NaN         NaN         NaN  
1         NaN         NaN         NaN         NaN         NaN  
2         NaN         NaN         NaN         NaN         NaN  
3         NaN         NaN         NaN         NaN         NaN  
4         NaN         NaN         NaN         NaN         NaN  

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------  

In [ ]:
# Step 4: Additional cleaning (safety check)

# Keep required columns
tags = tags[['userId', 'movieId', 'tag', 'date']]

# Remove missing values
tags.dropna(inplace=True)

# Reset index
tags.reset_index(drop=True, inplace=True)

print("After Final Cleaning:")
print(tags.head())

After Final Cleaning:
   userId  movieId                tag              date
0       2    89774          tom hardy  24-10-2015 19:33
1       2    89774       boxing story  24-10-2015 19:33
2       2   106782    martin scorsese  24-10-2015 19:30
3       2   106782  leonardo dicaprio  24-10-2015 19:30
4       2   106782              drugs  24-10-2015 19:30


In [ ]:
# Step 5: Create sentiment labels using TextBlob

def get_sentiment(text):
    """
    Convert text into sentiment label using polarity score
    """
    score = TextBlob(text).sentiment.polarity

    if score > 0:
        return "Positive"
    elif score < 0:
        return "Negative"
    else:
        return "Neutral"

# Apply function
tags['sentiment'] = tags['tag'].apply(get_sentiment)

print("Sentiment Labels Added:")
print(tags.head())

Sentiment Labels Added:
   userId  movieId                tag              date sentiment
0       2    89774          tom hardy  24-10-2015 19:33   Neutral
1       2    89774       boxing story  24-10-2015 19:33   Neutral
2       2   106782    martin scorsese  24-10-2015 19:30   Neutral
3       2   106782  leonardo dicaprio  24-10-2015 19:30   Neutral
4       2   106782              drugs  24-10-2015 19:30   Neutral


In [ ]:
# Step 6: Check how many tags fall into each category

print("Sentiment Distribution:")
print(tags['sentiment'].value_counts())

Sentiment Distribution:
sentiment
Neutral     2879
Positive     466
Negative     338
Name: count, dtype: int64


In [ ]:
# Step 7: Convert text into numerical features

vectorizer = TfidfVectorizer(stop_words='english')

# Feature matrix (X)
X = vectorizer.fit_transform(tags['tag'])

# Target variable (y)
y = tags['sentiment']

print("Feature Matrix Shape:", X.shape)

Feature Matrix Shape: (3683, 1673)


In [ ]:
# Step 8: Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training Size:", X_train.shape)
print("Testing Size:", X_test.shape)

Training Size: (2946, 1673)
Testing Size: (737, 1673)


In [ ]:
# Step 9: Train the ML model

model = LogisticRegression(max_iter=200)

# Train model
model.fit(X_train, y_train)

print("Model Training Completed")

Model Training Completed


In [ ]:
# Step 10: Evaluate performance

y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Model Accuracy:", accuracy)

# Detailed metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 0.9253731343283582

Classification Report:
              precision    recall  f1-score   support

    Negative       1.00      0.58      0.74        77
     Neutral       0.91      1.00      0.95       573
    Positive       0.98      0.75      0.85        87

    accuracy                           0.93       737
   macro avg       0.97      0.78      0.85       737
weighted avg       0.93      0.93      0.92       737



In [ ]:
# Step 11: Function for predicting new sentiment

def predict_sentiment(text):
    vec = vectorizer.transform([text])
    return model.predict(vec)[0]

# Example predictions
print(predict_sentiment("amazing movie"))
print(predict_sentiment("boring and slow"))
print(predict_sentiment("average film"))

Neutral
Negative
Neutral


In [ ]:
# Step 12: Apply model to full dataset

tags['predicted_sentiment'] = model.predict(X)

print("Final Data:")
print(tags.head())

Final Data:
   userId  movieId                tag              date sentiment  \
0       2    89774          tom hardy  24-10-2015 19:33   Neutral   
1       2    89774       boxing story  24-10-2015 19:33   Neutral   
2       2   106782    martin scorsese  24-10-2015 19:30   Neutral   
3       2   106782  leonardo dicaprio  24-10-2015 19:30   Neutral   
4       2   106782              drugs  24-10-2015 19:30   Neutral   

  predicted_sentiment  
0             Neutral  
1             Neutral  
2             Neutral  
3             Neutral  
4             Neutral  


In [ ]:
# Step 13: Save results (DOES NOT replace original file)

tags.to_csv("sentiment_results.csv", index=False)

print("File saved as sentiment_results.csv")

File saved as sentiment_results.csv


In [ ]:
# Step 13: Save results (DOES NOT replace original file)

tags.to_csv("sentiment_results.csv", index=False)

print("File saved as sentiment_results.csv")

File saved as sentiment_results.csv
